# Disaggregated Prefill / Decode on 2× Free T4s (Kaggle)

**Reproducing the KV-cache handoff from Figure 15-4 (llm-d disaggregated architecture) on free hardware, with the real run's outputs preserved.**

One vLLM server does **prefill** (builds the KV cache) on GPU 0, a second does **decode** (generates tokens) on GPU 1, and the KV cache is transferred between them over **LMCache + NIXL**, the mechanism the book depicts.

### What this is (and isn't)
- **Full llm-d** is a *Kubernetes* app requiring NVIDIA GPUs (gateway, scheduler, variant autoscaler). It does **not** run on a Mac or a notebook.
- **This notebook** reproduces the *core mechanism* (the prefill→decode KV handoff) on **two free Kaggle T4 GPUs**.

### Environment
- Kaggle Notebook, accelerator = **GPU T4 ×2**, **Internet = ON**
- vLLM `0.26.0`, LMCache `0.5.2`, NIXL transport

> **How to read this:** every step has an explanation above it. **⚠️ ERROR WE HIT** cells document a real
> failure, its cause, and the fix. The code cells retain **the actual outputs from the real run** (long
> logs trimmed head+tail). That debugging trail is the point of the notebook.

---
## 1. Verify the two GPUs
The whole point is *two* GPUs. If you see only one, set **Settings → Accelerator → GPU T4 x2**.

In [101]:
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d4085e1c-f4fe-3c0a-6079-5a7565fd221e)
GPU 1: Tesla T4 (UUID: GPU-2e77b2e5-79be-c41b-2dd4-18de810cf040)


## 2. Install vLLM

In [102]:
!pip install -q vllm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.


### ⚠️ ERROR WE HIT: a huge red wall of dependency conflicts after `pip install`
The install prints a scary block (`cudf-cu12 ... requires cuda-python<13.0 ... incompatible`, protobuf,
numba, etc). **This is NOT a failure.** vLLM pulled newer libs than Kaggle's preinstalled data-science
packages (cudf, bigframes) wanted, and you aren't using those. The only real tension is cuda-python
12.x→13.x, so a **Run → Restart Session** afterward is worth doing.
**Fix:** restart, then verify the import. Ignore the red.

In [103]:
import vllm
print(vllm.__version__)

0.26.0


---
## 3. First idea: the disk-based `SharedStorageConnector` (and why it fails)
The intuitive first attempt: prefill writes KV to a folder, decode reads it back. Kept here because the
**error is instructive**, it doesn't work on current vLLM.

In [104]:
%%writefile prefill.py
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # prefill → GPU 0

from vllm import LLM, SamplingParams
from vllm.config import KVTransferConfig

prompts = [
    "The capital of France is",
    "The largest planet in the solar system is",
]

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",           # ungated, ~3GB in fp16
    dtype="half",                                  # T4 is Turing — force fp16, not bf16
    enforce_eager=True,
    gpu_memory_utilization=0.8,
    max_model_len=2048,
    kv_transfer_config=KVTransferConfig(
        kv_connector="SharedStorageConnector",
        kv_role="kv_producer",
        kv_connector_extra_config={"shared_storage_path": "local_storage"},
    ),
)

llm.generate(prompts, SamplingParams(temperature=0, max_tokens=1))  # prefill only
with open("output.txt", "w") as f:
    f.writelines(p + "\n" for p in prompts)
print("PREFILL done on GPU 0 → KV cache in ./local_storage")

Overwriting prefill.py


In [105]:
%%writefile decode.py
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"          # decode → GPU 1

from vllm import LLM, SamplingParams
from vllm.config import KVTransferConfig

with open("output.txt") as f:
    prompts = [l.strip() for l in f]

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.8,
    max_model_len=2048,
    kv_transfer_config=KVTransferConfig(
        kv_connector="SharedStorageConnector",
        kv_role="kv_consumer",
        kv_connector_extra_config={"shared_storage_path": "local_storage"},
    ),
)

for out in llm.generate(prompts, SamplingParams(temperature=0, max_tokens=40)):
    print(out.prompt, "→", out.outputs[0].text)

Overwriting decode.py


In [106]:
!python prefill.py

INFO 08-02 00:20:15 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'kv_transfer_config': KVTransferConfig(kv_connector='SharedStorageConnector', engine_id='0b695660-9923-4ca4-9f37-ae4e858b8af7', kv_buffer_device='cuda', kv_buffer_size=1000000000.0, kv_role='kv_producer', kv_rank=None, kv_parallel_size=1, kv_ip='127.0.0.1', kv_port=14579, kv_connector_extra_config={'shared_storage_path': 'local_storage'}, kv_connector_module_path=None, enable_permute_local_kv=False, kv_load_failure_policy='fail'), 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 08-02 00:20:19 [model.py:623] Resolved architecture: Qwen2ForCausalLM
WARNING 08-02 00:20:19 [model.py:2123] Casting torch.bfloat16 to torch.float16.
INFO 08-02 00:20:19 [model.py:1788] Using max model len 2048
INFO 08-02 00:20:19 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 08-02 00:20:19 [vllm.py:11

### ⚠️ ERROR WE HIT: `ValueError: Unsupported connector type: SharedStorageConnector`
**Cause:** the KV-connector API churns across vLLM versions. In `0.26.0` the disk-based
`SharedStorageConnector` was **removed entirely**. The online example was written for an older vLLM.

### Discover which connectors your build actually supports

In [107]:
p = "/usr/local/lib/python3.12/dist-packages/vllm/distributed/kv_transfer/kv_connector/v1/"
!grep -rhn "register\|_KV_CONNECTOR\|class.*Connector" {p}base.py {p}example_connector.py {p}simple_cpu_offload_connector.py

124:class KVConnectorRole(enum.Enum):
132:class KVConnectorHandshakeMetadata(ABC):  # noqa: B024
141:class KVConnectorMetadata(ABC):  # noqa: B024
150:class KVConnectorWorkerMetadata(ABC):
171:class KVConnectorBase_V1(ABC):
251:    def register_kv_caches(self, kv_caches: dict[str, torch.Tensor]):
253:        Initialize with the KV caches. Useful for pre-registering the
261:    def register_cross_layers_kv_cache(
270:        {register_kv_caches, register_cross_layers_kv_cache} will be called.
648:        registered connectors to return their own KVConnectorStats object,
690:        Create a KVConnectorPromMetrics subclass which should register
68:class ExampleConnectorMetadata(KVConnectorMetadata):
84:class ExampleConnector(KVConnectorBase_V1):
45:class SimpleCPUOffloadConnector(KVConnectorBase_V1, SupportsHMA):
120:    def register_kv_caches(self, kv_caches: dict[str, torch.Tensor]) -> None:
122:            self.worker_handler.register_kv_caches(kv_caches)


In [108]:
import vllm, os
factory = os.path.join(os.path.dirname(vllm.__file__),
                       "distributed/kv_transfer/kv_connector/factory.py")
!sed -n '150,245p' {factory}

# only load the files corresponding to the current connector.

KVConnectorFactory.register_connector(
    "ExampleConnector",
    "vllm.distributed.kv_transfer.kv_connector.v1.example_connector",
    "ExampleConnector",
)

KVConnectorFactory.register_connector(
    "ExampleHiddenStatesConnector",
    "vllm.distributed.kv_transfer.kv_connector.v1.example_hidden_states_connector",
    "ExampleHiddenStatesConnector",
)

KVConnectorFactory.register_connector(
    "LMCacheConnectorV1",
    "vllm.distributed.kv_transfer.kv_connector.v1.lmcache_connector",
    "LMCacheConnectorV1",
)

KVConnectorFactory.register_connector(
    "LMCacheMPConnector",
    "vllm.distributed.kv_transfer.kv_connector.v1.lmcache_mp_connector",
    "LMCacheMPConnector",
)

KVConnectorFactory.register_connector(
    "NixlConnector",
    "vllm.distributed.kv_transfer.kv_connector.v1.nixl",
    "NixlConnector",
)

KVConnectorFactory.register_connector(
    "NixlPullConnector",
    "vllm.distributed.kv_transfer.kv_connec

**Registry finding (vLLM 0.26.0):** no disk connector survives. Real options: `NixlConnector` (+Pull/Push),
`LMCacheConnectorV1`, `MooncakeConnector`, `MoRIIOConnector`, `SimpleCPUOffloadConnector`, `ExampleConnector`.
With two real GPUs we go for genuine GPU↔GPU transfer.

---
## 4. Second attempt: raw `NixlConnector` with two live servers
NIXL is the book's transport. First confirm the Python binding installs (the usual failure point).

In [109]:
!pip install -q nixl 2>&1 | tail -5

Launch prefill (producer, GPU 0:8100) and decode (consumer, GPU 1:8200) with `NixlConnector`.
**Distinct `handshake_port`/`kv_port` per side**, see the collision error below.

In [111]:
import subprocess, os
env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
env["VLLM_NIXL_SIDE_CHANNEL_PORT"] = "5600" 
prefill = subprocess.Popen(
    ["vllm","serve","Qwen/Qwen2.5-1.5B-Instruct","--port","8100",
     "--dtype","half","--enforce-eager","--gpu-memory-utilization","0.8","--max-model-len","2048",
     "--kv-transfer-config",
     '{"kv_connector":"NixlConnector","kv_role":"kv_producer","kv_port":14579,"kv_connector_extra_config":{"handshake_port":5600}}'],
    env=env, stdout=open("prefill.log","w"), stderr=subprocess.STDOUT)
print("prefill pid", prefill.pid)

prefill pid 3826


In [112]:
env2 = os.environ.copy(); env2["CUDA_VISIBLE_DEVICES"] = "1"
env2["VLLM_NIXL_SIDE_CHANNEL_PORT"] = "5601" 
decode = subprocess.Popen(
    ["vllm","serve","Qwen/Qwen2.5-1.5B-Instruct","--port","8200",
     "--dtype","half","--enforce-eager","--gpu-memory-utilization","0.8","--max-model-len","2048",
     "--kv-transfer-config",
     '{"kv_connector":"NixlConnector","kv_role":"kv_consumer","kv_port":14580,"kv_connector_extra_config":{"handshake_port":5601}}'],
    env=env2, stdout=open("decode.log","w"), stderr=subprocess.STDOUT)
print("decode pid", decode.pid)

decode pid 3832


### ⚠️ ERROR WE HIT #1: `zmq.error.ZMQError: Address already in use (5600)`
Read *past* the outer `Engine core initialization failed`. Real cause: both NIXL peers tried to bind the
same handshake port (5600); normally they're on separate nodes. **But the same log proves NIXL
*initialized* inside Kaggle's sandbox** (`Backend UCX was instantiated`, `Initialized NIXL agent`,
`GPU KV cache size: 317,952 tokens`). **Fix:** distinct `handshake_port` (5600/5601) + `kv_port`
(14579/14580). *Ignore* the `libnvrtc.so.13` (DeepGEMM) and `FA2 compute capability` (T4 pre-Ampere) noise.

In [ ]:
import time, requests
def wait(port, name, timeout=300):
    t0=time.time()
    while time.time()-t0<timeout:
        try:
            if requests.get(f"http://localhost:{port}/v1/models",timeout=2).ok:
                print(f"{name} up on :{port}"); return True
        except Exception: pass
        time.sleep(3)
    print(f"{name} did NOT come up"); return False
wait(8100,"prefill"); wait(8200,"decode")

prefill up on :8100
decode up on :8200


True

### ⚠️ ERROR WE HIT #2: the "it worked!" trap: `kv_transfer_params: None`
A hand-rolled proxy returned a *coherent* answer but `kv_transfer_params` was **None**, meaning decode
**re-ran the prompt locally** and ignored the handoff. **Two key lessons:** (1) **trust the logs, not the
output text**; (2) **don't hand-roll the proxy** vLLM ships the correct one.

In [ ]:
import requests

def disaggregated_generate(prompt, max_tokens=40):
    body = {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": 0,
    }

    # Stage 1 — PREFILL on GPU 0. max_tokens=1 forces prefill-only;
    # the response carries kv_transfer_params describing the KV NIXL staged.
    pf = requests.post("http://localhost:8100/v1/completions",
                       json={**body, "max_tokens": 1}).json()
    kv = pf["choices"][0].get("kv_transfer_params") or pf.get("kv_transfer_params")
    print("KV handoff metadata from prefill:", kv)

    # Stage 2 — DECODE on GPU 1, consuming the KV NIXL transferred.
    dec = requests.post("http://localhost:8200/v1/completions",
                        json={**body, "kv_transfer_params": kv}).json()
    return dec["choices"][0]["text"]

print(disaggregated_generate("The capital of France is"))

KV handoff metadata from prefill: None
 Paris. The capital of Italy is Rome. What is the capital of Spain?
A) Madrid
B) Barcelona
C) Seville
D) Valencia

To determine the capital of Spain,


In [ ]:
!grep -iE "nixl|remote|recv|pull|transfer|handshake" decode.log | tail -25

(APIServer pid=1172) INFO 08-01 23:25:18 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-1.5B-Instruct', 'port': 8200, 'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'enforce_eager': True, 'gpu_memory_utilization': 0.8, 'kv_transfer_config': KVTransferConfig(kv_connector='NixlConnector', engine_id='2c5c04a6-6825-498e-948d-1eb82b6ba7bc', kv_buffer_device='cuda', kv_buffer_size=1000000000.0, kv_role='kv_consumer', kv_rank=None, kv_parallel_size=1, kv_ip='127.0.0.1', kv_port=14580, kv_connector_extra_config={'handshake_port': 5601}, kv_connector_module_path=None, enable_permute_local_kv=False, kv_load_failure_policy='fail')}
(APIServer pid=1172) INFO 08-01 23:25:19 [nixl_utils.py:32] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
(APIServer pid=1172) INFO 08-01 23:25:19 [nixl_utils.py:68] NIXL is available
(EngineCore pid=1234) INFO 08-01 23:25:40 [core.py:116] Initializing a V1 LLM engine (v

---
## 5. The right approach: LMCache + NIXL (vLLM's *own* tested recipe)
LMCache orchestrates the handoff *over* NIXL, supplying the coordination the hand-rolled proxy lacked. Pull vLLM's
official example at your version tag.

In [ ]:
!git clone --depth 1 --branch v0.26.0 https://github.com/vllm-project/vllm.git /tmp/vsrc 2>/dev/null || git clone --depth 1 https://github.com/vllm-project/vllm.git /tmp/vsrc
!find /tmp/vsrc/examples -iname "*disagg*" -o -iname "*nixl*" | head -20

/tmp/vsrc/examples/disaggregated
/tmp/vsrc/examples/disaggregated/disaggregated_serving
/tmp/vsrc/examples/disaggregated/disaggregated_serving/disagg_proxy_multiturn.py
/tmp/vsrc/examples/disaggregated/disaggregated_serving/disagg_proxy_pushconnector_demo.py
/tmp/vsrc/examples/disaggregated/disaggregated_serving/disagg_proxy_demo.py
/tmp/vsrc/examples/disaggregated/disaggregated_encoder
/tmp/vsrc/examples/disaggregated/disaggregated_encoder/disagg_1e1pd_example.sh
/tmp/vsrc/examples/disaggregated/disaggregated_encoder/disagg_1e1p1d_example.sh
/tmp/vsrc/examples/disaggregated/disaggregated_encoder/disagg_epd_proxy.py
/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1
/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1/disagg_vllm_launcher.sh
/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1/disagg_proxy_server.py
/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1/disagg_example_nixl.sh


Read the recipe: the launcher holds the exact per-server flags/env.

In [ ]:
print(open("/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1/disagg_vllm_launcher.sh").read())

#!/bin/bash

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

if [[ $# -lt 1 ]]; then
    echo "Usage: $0 <prefiller | decoder> [model]"
    exit 1
fi

if [[ $# -eq 1 ]]; then
    echo "Using default model: meta-llama/Llama-3.1-8B-Instruct"
    MODEL="meta-llama/Llama-3.1-8B-Instruct"
else
    echo "Using model: $2"
    MODEL=$2
fi

# The prefillers and decoders in LMCache use the same hash seed for all chunk keys.
# This seed must be aligned so that decoders can identify and retrieve KV cache
# entries stored by prefillers.
#
# WARNING: Using a fixed hash seed is insecure and makes the application vulnerable to
# denial-of-service attacks. In a production environment, this should be set to a
# secure random value. This is set to a fixed value for demonstration purposes only.
export PYTHONHASHSEED=${VLLM_PYTHON_HASH_SEED:-123}

if [[ $1 == "prefiller" ]]; then
    # Prefiller listens on port 8100
    prefill_config_file=$SCRIPT_DIR/configs/lmcache-prefiller-config.yaml

    

**Launcher taught us:** use `LMCacheConnectorV1` (NIXL is the transport underneath);
`UCX_TLS=cuda_ipc,cuda_copy,tcp`; **fixed `PYTHONHASHSEED` on both** (decoder finds KV by hashing chunk
keys, so they must match); distinct `lmcache_rpc_port` per role.

In [ ]:
!pip install -q lmcache 2>&1 | tail -15

google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.0.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cudf-cu12 26

In [ ]:
!python -c "import lmcache; print('lmcache', lmcache.__version__)" 2>&1 | tail -3

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'lmcache'


---
## 6. The config schema migration

### ⚠️ ERROR WE HIT: `LMCache is unhealthy` because the config keys were silently ignored
Startup logged `Unknown configuration key: enable_nixl / nixl_role / nixl_peer_host`. The example YAML
used an **old LMCache schema**, so every NIXL setting was ignored and LMCache came up with **no peer**
(`enable_pd: False`). On a real request it flipped to `unhealthy` and ran local (`hit tokens: 0`).
**Fix:** read the installed library's schema and migrate keys; the rename map is in `config.py`.

In [ ]:
import lmcache, os
nc = os.path.join(os.path.dirname(lmcache.__file__), "v1/transfer_channel/nixl_channel.py")
print(nc)
!grep -nE "peer_host|peer_init|peer_alloc|peer_query|proxy|role|sender|receiver|connect|bind|listen|async init|def __init__|assert|timeout|host|port" {nc} | head -60

/usr/local/lib/python3.12/dist-packages/lmcache/v1/transfer_channel/nixl_channel.py
3:from dataclasses import dataclass
4:from typing import TYPE_CHECKING, Any, Optional, Union
5:import asyncio
6:import threading
7:import time
8:import uuid
11:import msgspec
12:import zmq
15:from lmcache.logging import init_logger
16:from lmcache.v1.memory_management import (
22:    from nixl._api import NixlAgent
25:from lmcache.v1.rpc_utils import get_zmq_context, get_zmq_socket
26:from lmcache.v1.transfer_channel.abstract import BaseTransferChannel
27:from lmcache.v1.transfer_channel.transfer_utils import (
43:    local_meta_bytes: bytes  # Metadata from the sender nixl agent
54:    remote_meta_bytes: bytes  # Metadata from the receiver nixl agent
58:    remote_xfer_dlist_bytes: bytes  # Serialized transfer descriptors for the receiver
67:    def __init__(
73:        assert "role" in kwargs
74:        assert "buffer_ptr" in kwargs
75:        assert "buffer_size" in kwargs
76:        assert "align_by

**Migration (old → new), from `config.py`:** `enable_nixl`→`enable_pd`, `nixl_role`→`pd_role`
(`"sender"`/`"receiver"`), `nixl_peer_host`→`pd_peer_host`, plus **three** ports
`pd_peer_init_port` / `pd_peer_alloc_port` / `pd_peer_query_port`, and `transfer_channel: "nixl"`.
Validation **requires** `pd_role`, `pd_buffer_size`, `pd_buffer_device`; the **receiver also asserts
`pd_peer_host is not None`**; PD auto-sets `save_unfull_chunk=True`.

In [ ]:
prefiller_yaml = """chunk_size: 256
local_cpu: False
max_local_cpu_size: 0

enable_pd: True
transfer_channel: "nixl"
pd_role: "sender"
pd_peer_host: "localhost"
pd_peer_init_port: 7300
pd_peer_alloc_port: 7301
pd_peer_query_port: 7302
pd_buffer_size: 1073741824
pd_buffer_device: "cuda"
pd_backend_mode: "async"
pd_skip_proxy_notification: True
"""

# RECEIVER: no pd_peer_host → it BINDS/listens on these ports
decoder_yaml = """chunk_size: 256
local_cpu: False
max_local_cpu_size: 0

enable_pd: True
transfer_channel: "nixl"
pd_role: "receiver"
pd_peer_init_port: 7300
pd_peer_alloc_port: 7301
pd_peer_query_port: 7302
pd_buffer_size: 1073741824
pd_buffer_device: "cuda"
pd_backend_mode: "async"
pd_skip_proxy_notification: True
"""

open("/kaggle/working/cfg/lmcache-prefiller-config.yaml","w").write(prefiller_yaml)
open("/kaggle/working/cfg/lmcache-decoder-config.yaml","w").write(decoder_yaml)
print("SENDER connects to localhost:7300-2, RECEIVER binds them")
print("--- decoder (should have NO pd_peer_host) ---")
print(open("/kaggle/working/cfg/lmcache-decoder-config.yaml").read())

SENDER connects to localhost:7300-2, RECEIVER binds them
--- decoder (should have NO pd_peer_host) ---
chunk_size: 256
local_cpu: False
max_local_cpu_size: 0

enable_pd: True
transfer_channel: "nixl"
pd_role: "receiver"
pd_peer_init_port: 7300
pd_peer_alloc_port: 7301
pd_peer_query_port: 7302
pd_buffer_size: 1073741824
pd_buffer_device: "cuda"
pd_backend_mode: "async"
pd_skip_proxy_notification: True



---
## 7. Clean GPU reset (before every relaunch)

### ⚠️ ERROR WE HIT: `Free memory on device cuda:0 (3.7/14.56 GiB) ... less than desired` (OOM)
`kill -9` on a vLLM process does **not** immediately free its GPU memory; killed workers strand CUDA
allocations, so the next launch OOMs. **Fix:** kill every PID `nvidia-smi` shows on the GPU, wait,
confirm memory returns to ~14.9 GiB, and drop `--gpu-memory-utilization` 0.8→0.6 for LMCache's PD buffer.

In [ ]:
import subprocess, time
for pid in subprocess.run(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader"],
                          capture_output=True,text=True).stdout.split():
    subprocess.run(["kill","-9",pid.strip()])
subprocess.run(["pkill","-9","-f","vllm"]); subprocess.run(["pkill","-9","-f","disagg_proxy"])
time.sleep(10)
!rm -f prefiller.log decoder.log proxy.log
print("free:", subprocess.run(["nvidia-smi","--query-gpu=memory.free","--format=csv,noheader"],capture_output=True,text=True).stdout.strip())

free: 14912 MiB
14912 MiB


---
## 8. Launch the disaggregated pipeline
**Order matters:** decoder/receiver **first** (it listens), then prefiller/sender, then proxy.
Both use `--gpu-memory-utilization 0.6` and the PD YAMLs.

### decoder (GPU 1, receiver). Run FIRST.

In [ ]:
import subprocess, os
env2 = os.environ.copy()
env2.update({
    "CUDA_VISIBLE_DEVICES": "1",
    "UCX_TLS": "cuda_ipc,cuda_copy,tcp",
    "PYTHONHASHSEED": "123",
    "VLLM_ENABLE_V1_MULTIPROCESSING": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "LMCACHE_CONFIG_FILE": "/kaggle/working/cfg/lmcache-decoder-config.yaml",
})
decoder = subprocess.Popen(
    ["vllm","serve","Qwen/Qwen2.5-1.5B-Instruct","--port","8200",
     "--dtype","half","--enforce-eager","--gpu-memory-utilization","0.6","--max-model-len","2048",
     "--kv-transfer-config",
     '{"kv_connector":"LMCacheConnectorV1","kv_role":"kv_consumer","kv_connector_extra_config":{"discard_partial_chunks":false,"lmcache_rpc_port":"consumer1"}}'],
    env=env2, stdout=open("decoder.log","w"), stderr=subprocess.STDOUT)
print("decoder pid", decoder.pid)

decoder pid 3418


In [ ]:
import time, requests
def wait(port, name, timeout=420):
    t0=time.time()
    while time.time()-t0<timeout:
        try:
            if requests.get(f"http://localhost:{port}/v1/models",timeout=2).ok:
                print(f"{name} up on :{port}"); return True
        except Exception: pass
        time.sleep(4)
    print(f"{name} NOT up — tail its log"); return False
wait(8200,"decoder")

decoder up on :8200


True

### prefiller (GPU 0, sender). Run AFTER decoder is up.

In [ ]:
env = os.environ.copy()
env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "UCX_TLS": "cuda_ipc,cuda_copy,tcp",
    "PYTHONHASHSEED": "123",
    "VLLM_ENABLE_V1_MULTIPROCESSING": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "LMCACHE_CONFIG_FILE": "/kaggle/working/cfg/lmcache-prefiller-config.yaml",
})
prefiller = subprocess.Popen(
    ["vllm","serve","Qwen/Qwen2.5-1.5B-Instruct","--port","8100",
     "--dtype","half","--enforce-eager","--gpu-memory-utilization","0.6","--max-model-len","2048",
     "--kv-transfer-config",
     '{"kv_connector":"LMCacheConnectorV1","kv_role":"kv_producer","kv_connector_extra_config":{"discard_partial_chunks":false,"lmcache_rpc_port":"producer1"}}'],
    env=env, stdout=open("prefiller.log","w"), stderr=subprocess.STDOUT)
print("prefiller pid", prefiller.pid)

prefiller pid 3587


In [ ]:
wait(8100,"prefiller")

prefiller up on :8100


True

### LMCache proxy on :9000. Run AFTER both are up.
(`404` on `GET /v1/models` is fine, since the proxy only serves `POST /v1/completions`; `Connection refused` is the problem.)

In [ ]:
!pkill -9 -f disagg_proxy
import time; time.sleep(2)
proxy = subprocess.Popen(
    ["python3","/tmp/vsrc/examples/disaggregated/lmcache/disagg_prefill_lmcache_v1/disagg_proxy_server.py",
     "--host","localhost","--port","9000",
     "--prefiller-host","localhost","--prefiller-port","8100",
     "--decoder-host","localhost","--decoder-port","8200"],
    env=os.environ.copy(), stdout=open("proxy.log","w"), stderr=subprocess.STDOUT)
time.sleep(8)
print("proxy pid", proxy.pid)
try:
    print("proxy reachable, status:", requests.get("http://localhost:9000/v1/models",timeout=5).status_code)
except Exception as e:
    print("proxy not up:", e); print(open("proxy.log").read()[-1200:])

proxy pid 3758
proxy reachable, status: 404


### Confirm config accepted + handshake (before sending traffic)
Win signal: **no `Unknown config` warnings**; dump shows `enable_pd: True`, `pd_role`,
`transfer_channel: nixl`, three ports; receiver logs `Initialized NIXL agent` + `async initialization loop`.

In [ ]:
!echo "== SENDER =="; grep -iE "connect|peer|nixl|bind|listen|unhealthy|handshake" prefiller.log | tail -12
!echo "== RECEIVER =="; grep -iE "bind|listen|peer|nixl|connect|unhealthy|init loop" decoder.log | tail -12

== SENDER ==
(EngineCore pid=3619) INFO 08-02 00:06:07 [utils.py:49] Connectors do not specify a kv cache layout, defaulting to NHD.
(EngineCore pid=3619) INFO 08-02 00:06:10 [factory.py:62] Creating v1 connector with name: LMCacheConnectorV1 and engine_id: 0182de29-b510-4bcc-923c-27e4a62a3f90
(EngineCore pid=3619) WARNING 08-02 00:06:10 [base.py:190] Initializing KVConnectorBase_V1. This API is experimental and subject to change in the future as we iterate the design.
(EngineCore pid=3619) INFO 08-02 00:06:10 [lmcache_connector.py:105] Initializing latest dev LMCache connector
(EngineCore pid=3619) [2026-08-02 00:06:10,951] LMCache WARNING: PD (Peer-to-Peer Disaggregation) requires save_unfull_chunk=True for complete KV cache transfer. Automatically setting save_unfull_chunk=True. (config.py:761:lmcache.v1.config)
(EngineCore pid=3619) [2026-08-02 00:06:10,956] LMCache INFO: Creating LMCacheEngine with config: {'chunk_size': 256, 'local_cpu': False, 'max_local_cpu_size': 0.0, 'local_c

### ⚠️ ERROR WE HIT: `LMCache hit tokens: 0` on a short prompt (chunk-size threshold)
A 5-token prompt (`"The capital of France is"`) transferred nothing: `chunk_size: 256` means LMCache
won't stage a handoff below one full chunk. **Fix:** send a **long** (>256-token) prompt (vLLM's own
benchmark uses `--random-input-len 7500`).

## 9. The test request + verdict
Send **one** long prompt through the proxy, then read the **logs**, not the answer.

In [ ]:
import requests
long_prompt = "The history of France is long and storied. " * 200
r = requests.post("http://localhost:9000/v1/completions",
    json={"model":"Qwen/Qwen2.5-1.5B-Instruct",
          "prompt": long_prompt + " In one sentence, summarize:",
          "max_tokens":40,"temperature":0}, timeout=240)
print("STATUS", r.status_code)
print(r.json()["choices"][0]["text"][:200])

STATUS 200
 "The history of France is long and storied." To provide a concise summary in one sentence, I'll say:

"The history of France spans centuries and is rich with tales, events, and influences


In [ ]:
!echo "===== PREFILLER (sender) ====="
!grep -iE "Reqid|hit tokens|need to|store|nixl|send|unhealthy" prefiller.log | tail -8
!echo "===== DECODER (receiver) ====="
!grep -iE "Reqid|hit tokens|need to|retriev|load|nixl|recv|unhealthy" decoder.log | tail -8

===== PREFILLER (sender) =====
(APIServer pid=3016) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore pid=3074) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore pid=3074) [2026-08-02 00:01:08,448] LMCache INFO: Creating LMCacheEngine with config: {'chunk_size': 256, 'local_cpu': False, 'max_local_cpu_size': 0.0, 'local_cpu_use_hugepages': False, 'reserve_local_cpu_size': 0.0, 'local_disk': None, 'local_disk_path_sharding': 'by_gpu', 'max_local_disk_size': 0.0, 'remote_url': None, 'remote_serde': 'naive', 'use_layerwise': False, 'save_decode_cache': False, 'pre_caching_hash_algorithm': 'builtin', 'enable_blending': False, 'blend_recompute_ratios': None, 'blend_thresholds': None, 'blend_check_layers': None, 'blend_min_tokens': 256, 'blend_special_str': ' # # ', 'retrieve_locations': None, '

### The win condition (how to read success)
**SUCCESS** = decoder `Reqid` shows large **`LMCache hit tokens: ~2000, need to load: ~2000`** with **no
`unhealthy`** → KV pulled over NIXL, GPU 0 → GPU 1. **Still `0` + `unhealthy`** = PD peers initialized but
the single-node handshake didn't complete (see next section).

---
## 10. Status, open issue, next steps
**Working:** two vLLM servers on two T4s; LMCache 0.5.2 PD mode **configured and accepted** (no
`Unknown config`); NIXL agents init on both sides; async channel starts; 2000-token request routed through
the official proxy to both engines.

**Open issue:** on a *single node*, the sender↔receiver PD handshake over NIXL doesn't always complete its
async init loop, so `hit tokens` can stay 0. PD mode is built for **two machines** (`pd_peer_host` = a
different box); one host with shared `localhost` ports is the seam.

**Two clean finishes:**
1. **`local_cpu` PD transport** instead of NIXL. KV still goes prefill→decode via host memory; more
   reliable on one node; proves the handoff (`hit tokens` lights up), just not "over NIXL".
2. **Two-node deployment** (two Modal GPU functions / RunPod boxes) where `pd_peer_host` is a real remote,
   the config LMCache actually tests. Everything here (schema, `save_unfull_chunk`, listener/dialer order)
   transfers unchanged.